In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
import joblib

In [2]:
df = pd.read_csv("PJME_preprocessed.csv",)

In [3]:
df.head()

,Datetime,PJME_MW,Hour,Day,DayOfWeek,Month,Year,Week,IsWeekend,Season,...,Month_sin,Month_cos,Lag_1,Lag_24,Lag_48,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,RollingStd_168
0,2002-01-08 01:00:00,29445.0,1,8,1,1,2002,2,0,Winter,...,0.5,0.866025,31187.0,26862.0,27100.0,30393.0,33560.208333,32513.869048,4425.965952,3861.770954
1,2002-01-08 02:00:00,28670.0,2,8,1,1,2002,2,0,Winter,...,0.5,0.866025,29445.0,25976.0,26097.0,29265.0,33672.458333,32510.327381,4256.159403,3865.039821
2,2002-01-08 03:00:00,28375.0,3,8,1,1,2002,2,0,Winter,...,0.5,0.866025,28670.0,25641.0,25793.0,28357.0,33786.375000,32510.434524,4064.104959,3864.924245
3,2002-01-08 04:00:00,28542.0,4,8,1,1,2002,2,0,Winter,...,0.5,0.866025,28375.0,25666.0,25657.0,27899.0,33906.208333,32514.261905,3851.076461,3860.646270
4,2002-01-08 05:00:00,29261.0,5,8,1,1,2002,2,0,Winter,...,0.5,0.866025,28542.0,26328.0,25778.0,28057.0,34028.416667,32521.428571,3640.941409,3853.433314


In [4]:
target = "PJME_MW"

In [5]:
df[target].describe()

count    145224.000000
mean      32078.417972
std        6466.680687
min       14544.000000
25%       27569.000000
50%       31419.000000
75%       35647.000000
max       62009.000000
Name: PJME_MW, dtype: float64

In [6]:
df[target].isnull().sum()

np.int64(0)

In [7]:
n = len(df)
test_size = int(n * 0.20)
test_start = n - test_size

In [8]:
validation_hours = 60 * 24
validation_start = (test_start - validation_hours)

In [9]:
train_df = df.iloc[:validation_start].copy()
validation_df = df.iloc[validation_start:test_start].copy()
test_df = df.iloc[test_start:].copy()

In [10]:
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (114740, 24)
Validation: (1440, 24)
Test: (29044, 24)


In [11]:
train_values = train_df[[target]].values
validation_values = validation_df[[target]].values
test_values = test_df[[target]].values

In [12]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_values)
validation_scaled = scaler.transform(validation_values)
test_scaled = scaler.transform(test_values)

In [13]:
print(
    train_scaled.min(),
    train_scaled.max()
)

0.0 1.0


In [14]:
LOOKBACKS = [24, 48, 168]
HORIZON = 24

In [15]:
def create_sequences(data,lookback,horizon):
    X = []
    y = []
    for i in range(lookback,len(data) - horizon + 1):
        X.append(data[i - lookback:i])
        y.append(data[i:i + horizon,0])
    return (np.array(X),np.array(y))

In [16]:
X_train_rnn_24, y_train_rnn_24 = create_sequences(
    train_scaled,
    24,
    24
)

In [17]:
print("X shape:", X_train_rnn_24.shape)
print("y shape:", y_train_rnn_24.shape)

X shape: (114693, 24, 1)
y shape: (114693, 24)


In [18]:
X_train_rnn_48, y_train_rnn_48 = create_sequences(
    train_scaled,
    48,
    24
)

In [19]:
print("X shape:", X_train_rnn_48.shape)
print("y shape:", y_train_rnn_48.shape)

X shape: (114669, 48, 1)
y shape: (114669, 24)


In [20]:
X_train_rnn_168, y_train_rnn_168 = create_sequences(
    train_scaled,
    168,
    24
)

In [21]:
print("X shape:", X_train_rnn_168.shape)
print("y shape:", y_train_rnn_168.shape)

X shape: (114549, 168, 1)
y shape: (114549, 24)


In [22]:
def build_rnn(lookback):
    model = Sequential([
        SimpleRNN(
            64,
            input_shape=(lookback, 1)
        ),
        Dense(
            64,
            activation="relu"
        ),
        Dense(24)
    ])
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )
    return model

In [23]:
rnn_24 = build_rnn(24)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [24]:
rnn_24.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,944 (38.84 KB)

 Trainable params: 9,944 (38.84 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
history_rnn_24 = rnn_24.fit(
    X_train_rnn_24,
    y_train_rnn_24,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0050 - mae: 0.0518 - val_loss: 0.0063 - val_mae: 0.0606
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0030 - mae: 0.0412 - val_loss: 0.0054 - val_mae: 0.0552
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0029 - mae: 0.0399 - val_loss: 0.0053 - val_mae: 0.0548
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0028 - mae: 0.0394 - val_loss: 0.0054 - val_mae: 0.0556
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0027 - mae: 0.0387 - val_loss: 0.0048 - val_mae: 0.0516
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0027 - mae: 0.0382 - val_loss: 0.0044 - val_mae: 0.0490
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0041 - val_mae: 0.0477
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0026 - mae: 0.0375 - val_loss: 0.0041 - val_mae: 0.0476
Epoch 9/15
1613/1613 ━━━━━━━━━━━━━━━━━━━

In [26]:
rnn_48 = build_rnn(48)

In [27]:
history_rnn_48 = rnn_48.fit(
    X_train_rnn_48,
    y_train_rnn_48,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0058 - mae: 0.0542 - val_loss: 0.0078 - val_mae: 0.0678
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0031 - mae: 0.0413 - val_loss: 0.0061 - val_mae: 0.0586
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0028 - mae: 0.0395 - val_loss: 0.0059 - val_mae: 0.0569
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0027 - mae: 0.0388 - val_loss: 0.0049 - val_mae: 0.0525
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0026 - mae: 0.0380 - val_loss: 0.0041 - val_mae: 0.0482
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0026 - mae: 0.0374 - val_loss: 0.0044 - val_mae: 0.0489
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0025 - mae: 0.0368 - val_loss: 0.0038 - val_mae: 0.0457
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0025 - mae: 0.0365 - val_loss: 0.0047 - val_mae: 0.0504
Epoch 9/15
1613/1613 ━━━━━━━━━━━━━━━━━━━

In [28]:
rnn_168 = build_rnn(168)

In [29]:
history_rnn_168 = rnn_168.fit(
    X_train_rnn_168,
    y_train_rnn_168,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0054 - mae: 0.0532 - val_loss: 0.0078 - val_mae: 0.0685
Epoch 2/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0030 - mae: 0.0405 - val_loss: 0.0064 - val_mae: 0.0605
Epoch 3/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0028 - mae: 0.0392 - val_loss: 0.0049 - val_mae: 0.0513
Epoch 4/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0027 - mae: 0.0388 - val_loss: 0.0054 - val_mae: 0.0548
Epoch 5/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0027 - mae: 0.0384 - val_loss: 0.0055 - val_mae: 0.0556
Epoch 6/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0026 - mae: 0.0379 - val_loss: 0.0054 - val_mae: 0.0566
Epoch 7/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0025 - mae: 0.0371 - val_loss: 0.0051 - val_mae: 0.0540
Epoch 8/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0055 - val_mae: 0.0559
Epoch 9/15
1611/1611 ━━━━━━━━━━━

In [30]:
def build_lstm(lookback):
    model = Sequential([
        LSTM(
            64,
            input_shape=(lookback, 1)
        ),
        Dense(
            64,
            activation="relu"
        ),
        Dense(24)
    ])
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )
    return model

In [31]:
X_train_lstm_24, y_train_lstm_24 = create_sequences(
    train_scaled,
    24,
    24
)

In [32]:
lstm_24 = build_lstm(24)
lstm_24.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,616 (88.34 KB)

 Trainable params: 22,616 (88.34 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
history_lstm_24 = lstm_24.fit(
    X_train_lstm_24,
    y_train_lstm_24,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0066 - mae: 0.0616 - val_loss: 0.0095 - val_mae: 0.0753
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0039 - mae: 0.0477 - val_loss: 0.0077 - val_mae: 0.0669
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0034 - mae: 0.0440 - val_loss: 0.0071 - val_mae: 0.0642
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0029 - mae: 0.0408 - val_loss: 0.0059 - val_mae: 0.0575
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0028 - mae: 0.0392 - val_loss: 0.0053 - val_mae: 0.0542
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0027 - mae: 0.0386 - val_loss: 0.0050 - val_mae: 0.0529
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0026 - mae: 0.0381 - val_loss: 0.0049 - val_mae: 0.0517
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0046 - val_mae: 0.0504
Epoch 9/15
1613/1613 ━━━━━━━━━━━━━━━━━━━

In [34]:
X_train_lstm_48, y_train_lstm_48 = create_sequences(
    train_scaled,
    48,
    24
)

In [35]:
lstm_48 = build_lstm(48)

In [36]:
history_lstm_48 = lstm_48.fit(
    X_train_lstm_48,
    y_train_lstm_48,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.0067 - mae: 0.0609 - val_loss: 0.0077 - val_mae: 0.0677
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0036 - mae: 0.0457 - val_loss: 0.0063 - val_mae: 0.0598
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0031 - mae: 0.0418 - val_loss: 0.0054 - val_mae: 0.0557
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0028 - mae: 0.0393 - val_loss: 0.0050 - val_mae: 0.0531
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0027 - mae: 0.0385 - val_loss: 0.0045 - val_mae: 0.0499
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0026 - mae: 0.0381 - val_loss: 0.0045 - val_mae: 0.0495
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0041 - val_mae: 0.0477
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0041 - val_mae: 0.0481
Epoch 9/15
1613/1613 ━━━━━━━━━━━

In [37]:
X_train_lstm_168, y_train_lstm_168 = create_sequences(
    train_scaled,
    168,
    24
)

In [38]:
lstm_168 = build_lstm(168)

In [39]:
history_lstm_168 = lstm_168.fit(
    X_train_lstm_168,
    y_train_lstm_168,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 45s 27ms/step - loss: 0.0068 - mae: 0.0607 - val_loss: 0.0077 - val_mae: 0.0673
Epoch 2/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 0.0034 - mae: 0.0442 - val_loss: 0.0075 - val_mae: 0.0665
Epoch 3/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 44s 27ms/step - loss: 0.0032 - mae: 0.0430 - val_loss: 0.0064 - val_mae: 0.0604
Epoch 4/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 44s 27ms/step - loss: 0.0030 - mae: 0.0411 - val_loss: 0.0060 - val_mae: 0.0578
Epoch 5/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 44s 27ms/step - loss: 0.0028 - mae: 0.0397 - val_loss: 0.0056 - val_mae: 0.0564
Epoch 6/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 0.0028 - mae: 0.0394 - val_loss: 0.0053 - val_mae: 0.0547
Epoch 7/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 44s 27ms/step - loss: 0.0027 - mae: 0.0385 - val_loss: 0.0047 - val_mae: 0.0514
Epoch 8/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 44s 27ms/step - loss: 0.0026 - mae: 0.0380 - val_loss: 0.0047 - val_mae: 0.0516
Epoch 9/15
1611/1611 ━━━

In [40]:
def build_gru(lookback):
    model = Sequential([
        GRU(
            64,
            input_shape=(lookback, 1)
        ),
        Dense(
            64,
            activation="relu"
        ),
        Dense(24)
    ])
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )
    return model

In [41]:
X_train_gru_24, y_train_gru_24 = create_sequences(
    train_scaled,
    24,
    24
)

In [42]:
gru_24 = build_gru(24)

In [43]:
history_gru_24 = gru_24.fit(
    X_train_gru_24,
    y_train_gru_24,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0072 - mae: 0.0637 - val_loss: 0.0091 - val_mae: 0.0731
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0039 - mae: 0.0483 - val_loss: 0.0078 - val_mae: 0.0669
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0036 - mae: 0.0460 - val_loss: 0.0068 - val_mae: 0.0618
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.0031 - mae: 0.0418 - val_loss: 0.0062 - val_mae: 0.0593
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0028 - mae: 0.0397 - val_loss: 0.0058 - val_mae: 0.0572
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0027 - mae: 0.0389 - val_loss: 0.0053 - val_mae: 0.0544
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0027 - mae: 0.0384 - val_loss: 0.0050 - val_mae: 0.0524
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0026 - mae: 0.0381 - val_loss: 0.0047 - val_mae: 0.0509
Epoch 9/15
1613/1613 ━━━━━━━━━━━━━━━━━━━

In [44]:
X_train_gru_48, y_train_gru_48 = create_sequences(
    train_scaled,
    48,
    24
)

In [45]:
gru_48 = build_gru(48)

In [46]:
history_gru_48 = gru_48.fit(
    X_train_gru_48,
    y_train_gru_48,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0071 - mae: 0.0626 - val_loss: 0.0081 - val_mae: 0.0684
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0036 - mae: 0.0460 - val_loss: 0.0069 - val_mae: 0.0618
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0032 - mae: 0.0422 - val_loss: 0.0063 - val_mae: 0.0603
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0028 - mae: 0.0395 - val_loss: 0.0060 - val_mae: 0.0586
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0028 - mae: 0.0393 - val_loss: 0.0056 - val_mae: 0.0571
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0028 - mae: 0.0394 - val_loss: 0.0056 - val_mae: 0.0572
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0028 - mae: 0.0398 - val_loss: 0.0055 - val_mae: 0.0573
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0028 - mae: 0.0397 - val_loss: 0.0051 - val_mae: 0.0546
Epoch 9/15
1613/1613 ━━━━━━━━━━━

In [47]:
X_train_gru_168, y_train_gru_168 = create_sequences(
    train_scaled,
    168,
    24
)

In [48]:
gru_168 = build_gru(168)

In [49]:
history_gru_168 = gru_168.fit(
    X_train_gru_168,
    y_train_gru_168,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 0.0067 - mae: 0.0607 - val_loss: 0.0072 - val_mae: 0.0644
Epoch 2/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 49s 31ms/step - loss: 0.0035 - mae: 0.0450 - val_loss: 0.0070 - val_mae: 0.0628
Epoch 3/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 0.0034 - mae: 0.0438 - val_loss: 0.0069 - val_mae: 0.0618
Epoch 4/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 0.0033 - mae: 0.0434 - val_loss: 0.0066 - val_mae: 0.0608
Epoch 5/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 0.0031 - mae: 0.0418 - val_loss: 0.0062 - val_mae: 0.0596
Epoch 6/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 0.0028 - mae: 0.0400 - val_loss: 0.0060 - val_mae: 0.0580
Epoch 7/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 0.0027 - mae: 0.0387 - val_loss: 0.0056 - val_mae: 0.0562
Epoch 8/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 50s 31ms/step - loss: 0.0027 - mae: 0.0385 - val_loss: 0.0045 - val_mae: 0.0507
Epoch 9/15
1611/1611 ━━━

In [50]:
def build_bilstm(lookback):
    model = Sequential([
        Bidirectional(
            LSTM(
                64
            ),
            input_shape=(lookback, 1)
        ),
        Dense(
            64,
            activation="relu"
        ),
        Dense(24)
    ])
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )
    return model

In [51]:
X_train_bilstm_24, y_train_bilstm_24 = create_sequences(
    train_scaled,
    24,
    24
)

In [52]:
bilstm_24 = build_bilstm(24)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [53]:
history_bilstm_24 = bilstm_24.fit(
    X_train_bilstm_24,
    y_train_bilstm_24,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.0060 - mae: 0.0577 - val_loss: 0.0078 - val_mae: 0.0671
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0032 - mae: 0.0429 - val_loss: 0.0050 - val_mae: 0.0530
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0029 - mae: 0.0401 - val_loss: 0.0047 - val_mae: 0.0510
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0027 - mae: 0.0391 - val_loss: 0.0044 - val_mae: 0.0494
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0027 - mae: 0.0383 - val_loss: 0.0043 - val_mae: 0.0487
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0045 - val_mae: 0.0499
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0025 - mae: 0.0373 - val_loss: 0.0046 - val_mae: 0.0510
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0025 - mae: 0.0369 - val_loss: 0.0048 - val_mae: 0.0524
Epoch 9/15
1613/1613 ━━━━━━━━━━━

In [54]:
X_train_bilstm_48, y_train_bilstm_48 = create_sequences(
    train_scaled,
    48,
    24
)

In [55]:
bilstm_48 = build_bilstm(48)

In [56]:
history_bilstm_48 = bilstm_48.fit(
    X_train_bilstm_48,
    y_train_bilstm_48,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0062 - mae: 0.0584 - val_loss: 0.0101 - val_mae: 0.0787
Epoch 2/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0034 - mae: 0.0444 - val_loss: 0.0068 - val_mae: 0.0623
Epoch 3/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0031 - mae: 0.0418 - val_loss: 0.0051 - val_mae: 0.0538
Epoch 4/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0029 - mae: 0.0404 - val_loss: 0.0049 - val_mae: 0.0534
Epoch 5/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0028 - mae: 0.0395 - val_loss: 0.0053 - val_mae: 0.0556
Epoch 6/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0027 - mae: 0.0389 - val_loss: 0.0052 - val_mae: 0.0553
Epoch 7/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0027 - mae: 0.0383 - val_loss: 0.0047 - val_mae: 0.0516
Epoch 8/15
1613/1613 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0026 - mae: 0.0380 - val_loss: 0.0045 - val_mae: 0.0507
Epoch 9/15
1613/1613 ━━━

In [57]:
X_train_bilstm_168, y_train_bilstm_168 = create_sequences(
    train_scaled,
    168,
    24
)

In [58]:
bilstm_168 = build_bilstm(168)

In [59]:
history_bilstm_168 = bilstm_168.fit(
    X_train_bilstm_168,
    y_train_bilstm_168,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    verbose=1
)

Epoch 1/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 88s 54ms/step - loss: 0.0061 - mae: 0.0571 - val_loss: 0.0082 - val_mae: 0.0702
Epoch 2/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 89s 55ms/step - loss: 0.0032 - mae: 0.0430 - val_loss: 0.0068 - val_mae: 0.0634
Epoch 3/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 90s 56ms/step - loss: 0.0028 - mae: 0.0398 - val_loss: 0.0061 - val_mae: 0.0603
Epoch 4/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 90s 56ms/step - loss: 0.0024 - mae: 0.0366 - val_loss: 0.0059 - val_mae: 0.0590
Epoch 5/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 89s 55ms/step - loss: 0.0023 - mae: 0.0356 - val_loss: 0.0057 - val_mae: 0.0577
Epoch 6/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 89s 55ms/step - loss: 0.0023 - mae: 0.0355 - val_loss: 0.0054 - val_mae: 0.0562
Epoch 7/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 90s 56ms/step - loss: 0.0023 - mae: 0.0355 - val_loss: 0.0056 - val_mae: 0.0572
Epoch 8/15
1611/1611 ━━━━━━━━━━━━━━━━━━━━ 90s 56ms/step - loss: 0.0023 - mae: 0.0356 - val_loss: 0.0051 - val_mae: 0.0542
Epoch 9/15
1611/1611 ━━━

In [60]:
def prepare_validation_data(train_scaled,validation_scaled,lookback,horizon):
    validation_input = np.concatenate([
        train_scaled[-lookback:],
        validation_scaled
    ])
    X_val, y_val = create_sequences(
        validation_input,
        lookback,
        horizon
    )
    return X_val, y_val

In [61]:
def prepare_test_data(validation_scaled,test_scaled,lookback,horizon):
    test_input = np.concatenate([
        validation_scaled[-lookback:],
        test_scaled
    ])
    X_test, y_test = create_sequences(
        test_input,
        lookback,
        horizon
    )
    return X_test, y_test

In [62]:
X_val_24, y_val_24 = prepare_validation_data(
    train_scaled,
    validation_scaled,
    24,
    24
)

In [63]:
X_val_48, y_val_48 = prepare_validation_data(
    train_scaled,
    validation_scaled,
    48,
    24
)

In [64]:
X_val_168, y_val_168 = prepare_validation_data(
    train_scaled,
    validation_scaled,
    168,
    24
)

In [65]:
X_test_24, y_test_24 = prepare_test_data(
    validation_scaled,
    test_scaled,
    24,
    24
)

In [66]:
X_test_48, y_test_48 = prepare_test_data(
    validation_scaled,
    test_scaled,
    48,
    24
)

In [67]:
X_test_168, y_test_168 = prepare_test_data(
    validation_scaled,
    test_scaled,
    168,
    24
)

In [68]:
def calculate_metrics(y_actual_scaled, y_pred_scaled, scaler):
    y_actual = scaler.inverse_transform(
        y_actual_scaled.reshape(-1, 1)
    ).reshape(y_actual_scaled.shape)

    y_pred = scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).reshape(y_pred_scaled.shape)
    
    actual = y_actual.flatten()
    predicted = y_pred.flatten()
    
    mae = mean_absolute_error(actual,predicted)
    mse = mean_squared_error(actual,predicted)
    rmse = np.sqrt(mse)
    non_zero = actual != 0
    mape = np.mean(
        np.abs(
            (actual[non_zero] - predicted[non_zero])
            / actual[non_zero]
        )
    ) * 100
    r2 = r2_score(actual,predicted)
    bias = np.mean(predicted - actual)
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [69]:
rnn_pred_24 = rnn_24.predict(
    X_val_24,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [70]:
rnn_24_metrics = calculate_metrics(
    y_val_24,
    rnn_pred_24,
    scaler
)

In [71]:
rnn_pred_48 = rnn_48.predict(
    X_val_48,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [72]:
rnn_48_metrics = calculate_metrics(
    y_val_48,
    rnn_pred_48,
    scaler
)

In [73]:
rnn_pred_168 = rnn_168.predict(
    X_val_168,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [74]:
rnn_168_metrics = calculate_metrics(
    y_val_168,
    rnn_pred_168,
    scaler
)

In [75]:
rnn_results = pd.DataFrame([
    {
        "Model": "RNN",
        "Lookback": "24 hours",
        **rnn_24_metrics
    },
    {
        "Model": "RNN",
        "Lookback": "48 hours",
        **rnn_48_metrics
    },
    {
        "Model": "RNN",
        "Lookback": "168 hours",
        **rnn_168_metrics
    }
])

In [76]:
best_rnn_r2 = rnn_results.loc[
    rnn_results["R2"].idxmax()
]
best_rnn_r2

Model                  RNN
Lookback          48 hours
MAE            2804.554276
MSE         13376792.71675
RMSE           3657.429797
MAPE              8.108753
R2                0.565176
Bias          -1811.251916
Name: 1, dtype: object

In [77]:
lstm_pred_24 = lstm_24.predict(
    X_val_24,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [78]:
lstm_24_metrics = calculate_metrics(
    y_val_24,
    lstm_pred_24,
    scaler
)

In [79]:
lstm_pred_48 = lstm_48.predict(
    X_val_48,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [80]:
lstm_48_metrics = calculate_metrics(
    y_val_48,
    lstm_pred_48,
    scaler
)

In [81]:
lstm_pred_168 = lstm_168.predict(
    X_val_168,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [82]:
lstm_168_metrics = calculate_metrics(
    y_val_168,
    lstm_pred_168,
    scaler
)

In [83]:
lstm_results = pd.DataFrame([
    {
        "Model": "LSTM",
        "Lookback": "24 hours",
        **lstm_24_metrics
    },
    {
        "Model": "LSTM",
        "Lookback": "48 hours",
        **lstm_48_metrics
    },
    {
        "Model": "LSTM",
        "Lookback": "168 hours",
        **lstm_168_metrics
    }
])

In [84]:
gru_pred_24 = gru_24.predict(
    X_val_24,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [85]:
gru_24_metrics = calculate_metrics(
    y_val_24,
    gru_pred_24,
    scaler
)

In [86]:
gru_pred_48 = gru_48.predict(
    X_val_48,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [87]:
gru_48_metrics = calculate_metrics(
    y_val_48,
    gru_pred_48,
    scaler
)

In [88]:
gru_pred_168 = gru_168.predict(
    X_val_168,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [89]:
gru_168_metrics = calculate_metrics(
    y_val_168,
    gru_pred_168,
    scaler
)

In [90]:
gru_results = pd.DataFrame([
    {
        "Model": "GRU",
        "Lookback": "24 hours",
        **gru_24_metrics
    },
    {
        "Model": "GRU",
        "Lookback": "48 hours",
        **gru_48_metrics
    },
    {
        "Model": "GRU",
        "Lookback": "168 hours",
        **gru_168_metrics
    }
])

In [91]:
bilstm_pred_24 = bilstm_24.predict(
    X_val_24,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [92]:
bilstm_24_metrics = calculate_metrics(
    y_val_24,
    bilstm_pred_24,
    scaler
)

In [93]:
bilstm_pred_48 = bilstm_48.predict(
    X_val_48,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [94]:
bilstm_48_metrics = calculate_metrics(
    y_val_48,
    bilstm_pred_48,
    scaler
)

In [95]:
bilstm_pred_168 = bilstm_168.predict(
    X_val_168,
    verbose=1
)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


In [96]:
bilstm_168_metrics = calculate_metrics(
    y_val_168,
    bilstm_pred_168,
    scaler
)

In [97]:
bilstm_results = pd.DataFrame([
    {
        "Model": "Bi-LSTM",
        "Lookback": "24 hours",
        **bilstm_24_metrics
    },
    {
        "Model": "Bi-LSTM",
        "Lookback": "48 hours",
        **bilstm_48_metrics
    },
    {
        "Model": "Bi-LSTM",
        "Lookback": "168 hours",
        **bilstm_168_metrics
    }
])

In [98]:
all_results = pd.concat([rnn_results,lstm_results,gru_results,bilstm_results],ignore_index=True)
all_results

,Model,Lookback,MAE,MSE,RMSE,MAPE,R2,Bias
0,RNN,24 hours,2934.691966,1.507680e+07,3882.885505,8.381019,0.509915,-2164.579669
1,RNN,48 hours,2804.554276,1.337679e+07,3657.429797,8.108753,0.565176,-1811.251916
2,RNN,168 hours,3091.808792,1.631260e+07,4038.886552,9.044437,0.469744,-1584.123233
3,LSTM,24 hours,2927.467063,1.455961e+07,3815.705578,8.338201,0.526727,-2346.953335
4,LSTM,48 hours,2975.498900,1.507415e+07,3882.544249,8.572470,0.510001,-2405.554376
5,LSTM,168 hours,2753.207870,1.250400e+07,3536.100119,8.019766,0.593546,-1905.002036
6,GRU,24 hours,2995.379049,1.501655e+07,3875.118779,8.514461,0.511874,-2440.578961
7,GRU,48 hours,2941.408518,1.399695e+07,3741.250010,8.494127,0.545017,-2401.765038
8,GRU,168 hours,2907.838106,1.410485e+07,3755.642477,8.411296,0.541509,-2263.152849
9,Bi-LSTM,24 hours,2991.414989,1.523597e+07,3903.328432,8.521060,0.504741,-2418.475490


In [99]:
all_results_sorted = all_results.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)
all_results_sorted

,Model,Lookback,MAE,MSE,RMSE,MAPE,R2,Bias
0,LSTM,168 hours,2753.207870,1.250400e+07,3536.100119,8.019766,0.593546,-1905.002036
1,RNN,48 hours,2804.554276,1.337679e+07,3657.429797,8.108753,0.565176,-1811.251916
2,Bi-LSTM,168 hours,2828.005018,1.353820e+07,3679.429539,8.128915,0.559929,-2301.413998
3,Bi-LSTM,48 hours,2826.661405,1.388515e+07,3726.277780,8.073060,0.548651,-2308.041491
4,GRU,48 hours,2941.408518,1.399695e+07,3741.250010,8.494127,0.545017,-2401.765038
5,GRU,168 hours,2907.838106,1.410485e+07,3755.642477,8.411296,0.541509,-2263.152849
6,LSTM,24 hours,2927.467063,1.455961e+07,3815.705578,8.338201,0.526727,-2346.953335
7,GRU,24 hours,2995.379049,1.501655e+07,3875.118779,8.514461,0.511874,-2440.578961
8,LSTM,48 hours,2975.498900,1.507415e+07,3882.544249,8.572470,0.510001,-2405.554376
9,RNN,24 hours,2934.691966,1.507680e+07,3882.885505,8.381019,0.509915,-2164.579669


In [100]:
best_r2_model = all_results.loc[
    all_results["R2"].idxmax()
]
print("Best R² Model:")
print(best_r2_model)

Best R² Model:
Model                  LSTM
Lookback          168 hours
MAE              2753.20787
MSE         12504004.051972
RMSE            3536.100119
MAPE               8.019766
R2                 0.593546
Bias           -1905.002036
Name: 5, dtype: object


In [101]:
all_results.to_csv(
    "phase5_validation_results.csv",
    index=False
)

In [102]:
rnn_test_pred_24 = rnn_24.predict(
    X_test_24,
    verbose=1
)
rnn_test_24_metrics = calculate_metrics(
    y_test_24,
    rnn_test_pred_24,
    scaler
)
rnn_test_24_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 0s 483us/step


{'MAE': 2192.2045358548057,
 'MSE': 9394085.244993657,
 'RMSE': np.float64(3064.977201382362),
 'MAPE': np.float64(6.66768725409914),
 'R2': 0.77683869378616,
 'Bias': np.float64(-1266.3813713600127)}

In [103]:
rnn_test_pred_48 = rnn_48.predict(
    X_test_48,
    verbose=1
)
rnn_test_48_metrics = calculate_metrics(
    y_test_48,
    rnn_test_pred_48,
    scaler
)
rnn_test_48_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 1s 691us/step


{'MAE': 2213.2959504186488,
 'MSE': 9109533.569353756,
 'RMSE': np.float64(3018.2003858845683),
 'MAPE': np.float64(6.799617667046816),
 'R2': 0.7835983645752854,
 'Bias': np.float64(-1181.4494048776776)}

In [104]:
rnn_test_pred_168 = rnn_168.predict(
    X_test_168,
    verbose=1
)
rnn_test_168_metrics = calculate_metrics(
    y_test_168,
    rnn_test_pred_168,
    scaler
)
rnn_test_168_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


{'MAE': 2291.991607342483,
 'MSE': 9981898.557856519,
 'RMSE': np.float64(3159.414274490846),
 'MAPE': np.float64(7.0557929776167665),
 'R2': 0.7628748874881207,
 'Bias': np.float64(-1125.9281247229467)}

In [105]:
lstm_test_pred_24 = lstm_24.predict(
    X_test_24,
    verbose=1
)
lstm_test_24_metrics = calculate_metrics(
    y_test_24,
    lstm_test_pred_24,
    scaler
)
lstm_test_24_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 1s 960us/step


{'MAE': 2147.7200362172634,
 'MSE': 8746457.619152661,
 'RMSE': np.float64(2957.441059286332),
 'MAPE': np.float64(6.476004461023138),
 'R2': 0.792223419723139,
 'Bias': np.float64(-1406.741873066908)}

In [106]:
lstm_test_pred_48 = lstm_48.predict(
    X_test_48,
    verbose=1
)
lstm_test_48_metrics = calculate_metrics(
    y_test_48,
    lstm_test_pred_48,
    scaler
)
lstm_test_48_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


{'MAE': 2197.602512055409,
 'MSE': 9036348.702802373,
 'RMSE': np.float64(3006.0520126575275),
 'MAPE': np.float64(6.69406773019315),
 'R2': 0.785336908562141,
 'Bias': np.float64(-1433.060297339126)}

In [107]:
lstm_test_pred_168 = lstm_168.predict(
    X_test_168,
    verbose=1
)
lstm_test_168_metrics = calculate_metrics(
    y_test_168,
    lstm_test_pred_168,
    scaler
)
lstm_test_168_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step


{'MAE': 2115.9646072002997,
 'MSE': 8056107.735370645,
 'RMSE': np.float64(2838.328334666489),
 'MAPE': np.float64(6.51734176933782),
 'R2': 0.8086230347778852,
 'Bias': np.float64(-1236.1437540649445)}

In [108]:
gru_test_pred_24 = gru_24.predict(
    X_test_24,
    verbose=1
)
gru_test_24_metrics = calculate_metrics(
    y_test_24,
    gru_test_pred_24,
    scaler
)
gru_test_24_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 1s 876us/step


{'MAE': 2179.665522315849,
 'MSE': 8961803.452506922,
 'RMSE': np.float64(2993.6271398600934),
 'MAPE': np.float64(6.590835387392949),
 'R2': 0.787107769161563,
 'Bias': np.float64(-1418.7281542414644)}

In [109]:
gru_test_pred_48 = gru_48.predict(
    X_test_48,
    verbose=1
)
gru_test_48_metrics = calculate_metrics(
    y_test_48,
    gru_test_pred_48,
    scaler
)
gru_test_48_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


{'MAE': 2265.5166172210606,
 'MSE': 9156163.937833779,
 'RMSE': np.float64(3025.915388412865),
 'MAPE': np.float64(6.917065498932766),
 'R2': 0.7824906362901094,
 'Bias': np.float64(-1648.304142832229)}

In [110]:
gru_test_pred_168 = gru_168.predict(
    X_test_168,
    verbose=1
)
gru_test_168_metrics = calculate_metrics(
    y_test_168,
    gru_test_pred_168,
    scaler
)
gru_test_168_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


{'MAE': 2152.241306874225,
 'MSE': 8527355.986932363,
 'RMSE': np.float64(2920.163691804342),
 'MAPE': np.float64(6.594571528951662),
 'R2': 0.7974282912103254,
 'Bias': np.float64(-1355.2850053232376)}

In [111]:
bilstm_test_pred_24 = bilstm_24.predict(
    X_test_24,
    verbose=1
)
bilstm_test_24_metrics = calculate_metrics(
    y_test_24,
    bilstm_test_pred_24,
    scaler
)
bilstm_test_24_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


{'MAE': 2197.8244845249005,
 'MSE': 8951125.841891797,
 'RMSE': np.float64(2991.8432181335634),
 'MAPE': np.float64(6.684352762330077),
 'R2': 0.787361421270307,
 'Bias': np.float64(-1461.2064563979982)}

In [112]:
bilstm_test_pred_48 = bilstm_48.predict(
    X_test_48,
    verbose=1
)
bilstm_test_48_metrics = calculate_metrics(
    y_test_48,
    bilstm_test_pred_48,
    scaler
)
bilstm_test_48_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2120.8114430948394,
 'MSE': 8348925.735943277,
 'RMSE': np.float64(2889.4507671776096),
 'MAPE': np.float64(6.455978814485315),
 'R2': 0.8016669932063508,
 'Bias': np.float64(-1297.3517127958212)}

In [113]:
bilstm_test_pred_168 = bilstm_168.predict(
    X_test_168,
    verbose=1
)
bilstm_test_168_metrics = calculate_metrics(
    y_test_168,
    bilstm_test_pred_168,
    scaler
)
bilstm_test_168_metrics

907/907 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step


{'MAE': 2183.012965117012,
 'MSE': 8645917.501651622,
 'RMSE': np.float64(2940.394106518992),
 'MAPE': np.float64(6.638091479102353),
 'R2': 0.7946118017064068,
 'Bias': np.float64(-1575.7729029300936)}

In [122]:
test_results = pd.DataFrame([
    {
        "Model": "RNN",
        "Lookback": "24 hours",
        **rnn_test_24_metrics
    },
    {
        "Model": "RNN",
        "Lookback": "48 hours",
        **rnn_test_48_metrics
    },
    {
        "Model": "RNN",
        "Lookback": "168 hours",
        **rnn_test_168_metrics
    },

    {
        "Model": "LSTM",
        "Lookback": "24 hours",
        **lstm_test_24_metrics
    },
    {
        "Model": "LSTM",
        "Lookback": "48 hours",
        **lstm_test_48_metrics
    },
    {
        "Model": "LSTM",
        "Lookback": "168 hours",
        **lstm_test_168_metrics
    },

    {
        "Model": "GRU",
        "Lookback": "24 hours",
        **gru_test_24_metrics
    },
    {
        "Model": "GRU",
        "Lookback": "48 hours",
        **gru_test_48_metrics
    },
    {
        "Model": "GRU",
        "Lookback": "168 hours",
        **gru_test_168_metrics
    },

    {
        "Model": "Bi-LSTM",
        "Lookback": "24 hours",
        **bilstm_test_24_metrics
    },
    {
        "Model": "Bi-LSTM",
        "Lookback": "48 hours",
        **bilstm_test_48_metrics
    },
    {
        "Model": "Bi-LSTM",
        "Lookback": "168 hours",
        **bilstm_test_168_metrics
    }
])
test_results

,Model,Lookback,MAE,MSE,RMSE,MAPE,R2,Bias
0,RNN,24 hours,2192.204536,9.394085e+06,3064.977201,6.667687,0.776839,-1266.381371
1,RNN,48 hours,2213.295950,9.109534e+06,3018.200386,6.799618,0.783598,-1181.449405
2,RNN,168 hours,2291.991607,9.981899e+06,3159.414274,7.055793,0.762875,-1125.928125
3,LSTM,24 hours,2147.720036,8.746458e+06,2957.441059,6.476004,0.792223,-1406.741873
4,LSTM,48 hours,2197.602512,9.036349e+06,3006.052013,6.694068,0.785337,-1433.060297
5,LSTM,168 hours,2115.964607,8.056108e+06,2838.328335,6.517342,0.808623,-1236.143754
6,GRU,24 hours,2179.665522,8.961803e+06,2993.627140,6.590835,0.787108,-1418.728154
7,GRU,48 hours,2265.516617,9.156164e+06,3025.915388,6.917065,0.782491,-1648.304143
8,GRU,168 hours,2152.241307,8.527356e+06,2920.163692,6.594572,0.797428,-1355.285005
9,Bi-LSTM,24 hours,2197.824485,8.951126e+06,2991.843218,6.684353,0.787361,-1461.206456


In [115]:
test_results_sorted = test_results.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)
test_results_sorted

,Model,Lookback,MAE,MSE,RMSE,MAPE,R2,Bias
0,LSTM,168 hours,2115.964607,8.056108e+06,2838.328335,6.517342,0.808623,-1236.143754
1,Bi-LSTM,48 hours,2120.811443,8.348926e+06,2889.450767,6.455979,0.801667,-1297.351713
2,GRU,168 hours,2152.241307,8.527356e+06,2920.163692,6.594572,0.797428,-1355.285005
3,Bi-LSTM,168 hours,2183.012965,8.645918e+06,2940.394107,6.638091,0.794612,-1575.772903
4,LSTM,24 hours,2147.720036,8.746458e+06,2957.441059,6.476004,0.792223,-1406.741873
5,Bi-LSTM,24 hours,2197.824485,8.951126e+06,2991.843218,6.684353,0.787361,-1461.206456
6,GRU,24 hours,2179.665522,8.961803e+06,2993.627140,6.590835,0.787108,-1418.728154
7,LSTM,48 hours,2197.602512,9.036349e+06,3006.052013,6.694068,0.785337,-1433.060297
8,RNN,48 hours,2213.295950,9.109534e+06,3018.200386,6.799618,0.783598,-1181.449405
9,GRU,48 hours,2265.516617,9.156164e+06,3025.915388,6.917065,0.782491,-1648.304143


In [116]:
validation_results = all_results.rename(
    columns={
        "MAE": "Val_MAE",
        "MSE": "Val_MSE",
        "RMSE": "Val_RMSE",
        "MAPE": "Val_MAPE",
        "R2": "Val_R2",
        "Bias": "Val_Bias"
    }
)

In [117]:
test_results_compare = test_results.rename(
    columns={
        "MAE": "Test_MAE",
        "MSE": "Test_MSE",
        "RMSE": "Test_RMSE",
        "MAPE": "Test_MAPE",
        "R2": "Test_R2",
        "Bias": "Test_Bias"
    }
)

In [118]:
final_comparison = pd.merge(
    validation_results,
    test_results_compare,
    on=["Model", "Lookback"]
)
final_comparison

,Model,Lookback,Val_MAE,Val_MSE,Val_RMSE,Val_MAPE,Val_R2,Val_Bias,Test_MAE,Test_MSE,Test_RMSE,Test_MAPE,Test_R2,Test_Bias
0,RNN,24 hours,2934.691966,1.507680e+07,3882.885505,8.381019,0.509915,-2164.579669,2192.204536,9.394085e+06,3064.977201,6.667687,0.776839,-1266.381371
1,RNN,48 hours,2804.554276,1.337679e+07,3657.429797,8.108753,0.565176,-1811.251916,2213.295950,9.109534e+06,3018.200386,6.799618,0.783598,-1181.449405
2,RNN,168 hours,3091.808792,1.631260e+07,4038.886552,9.044437,0.469744,-1584.123233,2291.991607,9.981899e+06,3159.414274,7.055793,0.762875,-1125.928125
3,LSTM,24 hours,2927.467063,1.455961e+07,3815.705578,8.338201,0.526727,-2346.953335,2147.720036,8.746458e+06,2957.441059,6.476004,0.792223,-1406.741873
4,LSTM,48 hours,2975.498900,1.507415e+07,3882.544249,8.572470,0.510001,-2405.554376,2197.602512,9.036349e+06,3006.052013,6.694068,0.785337,-1433.060297
5,LSTM,168 hours,2753.207870,1.250400e+07,3536.100119,8.019766,0.593546,-1905.002036,2115.964607,8.056108e+06,2838.328335,6.517342,0.808623,-1236.143754
6,GRU,24 hours,2995.379049,1.501655e+07,3875.118779,8.514461,0.511874,-2440.578961,2179.665522,8.961803e+06,2993.627140,6.590835,0.787108,-1418.728154
7,GRU,48 hours,2941.408518,1.399695e+07,3741.250010,8.494127,0.545017,-2401.765038,2265.516617,9.156164e+06,3025.915388,6.917065,0.782491,-1648.304143
8,GRU,168 hours,2907.838106,1.410485e+07,3755.642477,8.411296,0.541509,-2263.152849,2152.241307,8.527356e+06,2920.163692,6.594572,0.797428,-1355.285005
9,Bi-LSTM,24 hours,2991.414989,1.523597e+07,3903.328432,8.521060,0.504741,-2418.475490,2197.824485,8.951126e+06,2991.843218,6.684353,0.787361,-1461.206456


In [119]:
best_test_model = test_results.loc[
    test_results["RMSE"].idxmin()
]
print("Best Test Model:")
print(best_test_model)

Best Test Model:
Model                 LSTM
Lookback         168 hours
MAE            2115.964607
MSE         8056107.735371
RMSE           2838.328335
MAPE              6.517342
R2                0.808623
Bias          -1236.143754
Name: 5, dtype: object


In [120]:
best_test_r2 = test_results.loc[
    test_results["R2"].idxmax()
]
print("Best Test R²:")
print(best_test_r2)

Best Test R²:
Model                 LSTM
Lookback         168 hours
MAE            2115.964607
MSE         8056107.735371
RMSE           2838.328335
MAPE              6.517342
R2                0.808623
Bias          -1236.143754
Name: 5, dtype: object
